In [5]:
import xarray as xr 
import json
import numpy as np
import cmaps
import matplotlib.pyplot as plt
import pyicon as pyic
import subprocess
from pathlib import Path
import os

# Cartopy for geographic plotting
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cmap
# import xesmf as xe
import glob

from slurm_cluster import init_dask_slurm_cluster
client, cluster = init_dask_slurm_cluster(scale=2, processes=20, cores=100, memory="200GiB", walltime="00:10:00")

/home/m/m301254/.conda/envs/env02_waves/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 33927 instead
  warnings.warn(


#!/usr/bin/env bash

#SBATCH -J dask-worker
#SBATCH -e /scratch/m/m301250/dask_logs//dask-worker-%J.err
#SBATCH -o /scratch/m/m301250/dask_logs//dask-worker-%J.out
#SBATCH -p compute
#SBATCH -A mh0033
#SBATCH -n 1
#SBATCH --cpus-per-task=100
#SBATCH --mem=200G
#SBATCH -t 00:10:00

/home/m/m301254/.conda/envs/env02_waves/bin/python -m distributed.cli.dask_worker tcp://10.128.10.136:39989 --name dummy-name --nthreads 5 --memory-limit 10.00GiB --nworkers 20 --nanny --death-timeout 60 --local-directory /scratch/m/m301250/dask_temp/ --interface ib0



KeyboardInterrupt: 

In [1]:
import sys
print([p for p in sys.path if "m301254" in p])

['/home/m/m301254/.conda/envs/env02_waves/lib/python311.zip', '/home/m/m301254/.conda/envs/env02_waves/lib/python3.11', '/home/m/m301254/.conda/envs/env02_waves/lib/python3.11/lib-dynload', '/home/m/m301254/.conda/envs/env02_waves/lib/python3.11/site-packages']


In [ ]:
def mean_above_quantile(data, threshold):
    filtered = data[data >= threshold]
    return np.nanmean(filtered) if len(filtered) > 0 else np.nan

import dask
@dask.delayed
def process_lat_band(lat, pr, clat, cell_area):
    mask = (clat >= (lat - 0.5)) & (clat <= (lat + 0.5))
    pr_lat = pr.where(mask, drop=True)
    cell_area_lat = cell_area.where(mask, drop=True)

    pr_weighted = pr_lat * cell_area_lat
    quantiles = pr_weighted.quantile([0.9, 0.99, 0.999])

    # calculate mean above each quantile using apply_ufunc
    result = xr.apply_ufunc(
        mean_above_quantile,
        pr_weighted.stack(all=('time','ncells')),       # consider data at all time and ncells together
        quantiles,
        vectorize=True,
        input_core_dims=[['all'], []],
        output_core_dims=[[]],
    )

    return result.expand_dims(lat=[lat]).compute()


In [ ]:
# load grid data
ds_grid = xr.open_dataset('/work/mh0033/m301250/20251214_aqua-convection/data/icon_grid_0015_R02B09_G.nc')
clat = ds_grid['clat'].rename(cell='ncells') / np.pi * 180    # convert to degrees
clon = ds_grid['clon'].rename(cell='ncells') / np.pi * 180    # convert to degrees
cell_area = ds_grid['cell_area'].rename(cell='ncells')
cell_area_chunked = cell_area.chunk({'ncells': 10000000})
cell_area_chunked

<xarray.DataArray 'cell_area' (ncells: 20971520)> Size: 168MB
dask.array<xarray-<this-array>, shape=(20971520,), dtype=float64, chunksize=(10000000,), chunktype=numpy.ndarray>
Coordinates:
    clon     (ncells) float64 168MB dask.array<chunksize=(10000000,), meta=np.ndarray>
    clat     (ncells) float64 168MB dask.array<chunksize=(10000000,), meta=np.ndarray>
Dimensions without coordinates: ncells
Attributes:
    long_name:                    area of grid cell
    units:                        m2
    standard_name:                area
    grid_type:                    unstructured
    number_of_grid_in_reference:  1

In [ ]:
import glob

# CTRL run
files = glob.glob(
    "/work/bm1183/m301049/icon-mpim/experiments/jed0111/jed0111_atm_2d_daymean_*"
)
files.sort()

ds = xr.open_mfdataset(files[:5])
pr_chunked = ds['pr'].sel(time=ds['time.hour']==0).chunk({'ncells': 10000000})

lats = np.arange(-80, 80+1, 1)      # when lat=85 we have 17695 cells
delayed_results = [process_lat_band(lat, pr_chunked, clat, cell_area_chunked) for lat in lats]
result_CTRL_ls = dask.compute(*delayed_results)
result_CTRL_xr = xr.concat(result_CTRL_ls, dim='lat')

print("CTRL done")